# Kaggle Competition Strategies: What Grandmasters Actually Do

This notebook covers the real techniques that separate top Kaggle competitors from everyone else. These are exactly the strategies interviewers expect senior data scientists to know.

## What You Will Learn

1. **Competition Framework** - How grandmasters approach a new competition
2. **Feature Engineering Masterclass** - Squeezing every signal from the Titanic dataset
3. **Stacking and Blending** - Building meta-learners from scratch
4. **Hyperparameter Tuning at Scale** - From grid search to Bayesian optimization
5. **Real Competition Techniques** - Pseudo-labeling, TTA, rank averaging
6. **End-to-End Kaggle Workflow** - Full pipeline beating the baseline

> **Interview Tip**: When asked about your approach to a new ML problem, walk through these exact sections. It shows systematic thinking and practical experience.

---
# SECTION 1: Competition Framework

## How Kaggle Grandmasters Approach a New Problem

The single biggest mistake beginners make: they jump straight to modeling. Grandmasters spend **50-70% of their time on EDA and feature engineering** before fitting a single model.

### The Grandmaster Mental Model

```
1. Read the problem description 3 times
2. Understand the evaluation metric deeply
3. Look at data distributions and relationships
4. Form hypotheses about what drives the target
5. Build a solid validation framework FIRST
6. Start with simple baseline, iterate from there
7. Track every experiment (never trust your memory)
```

### The Most Important Rule
**Your local CV score must correlate with the leaderboard.** If it doesn't, everything else is noise.

In [ ]:
# Setup and imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score, cross_val_predict
)
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor, GradientBoostingRegressor
)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error,
    mean_absolute_error, log_loss
)
import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Seaborn: {sns.__version__}')

## 1.1 The 15-Point EDA Checklist

Most competitors miss critical insights because they do a shallow EDA. Here is the complete checklist used by top competitors.

In [ ]:
# Load Titanic dataset as our working example
df = sns.load_dataset('titanic')
print('Titanic dataset loaded.')
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')

In [ ]:
# EDA CHECKLIST POINT 1: Basic shape and dtypes
print('=== EDA CHECKLIST ===')
print('\n[1] Dataset shape and dtypes')
print(df.dtypes)
print(f'\nTotal rows: {len(df):,}')
print(f'Total columns: {df.shape[1]}')

In [ ]:
# EDA CHECKLIST POINT 2: Missing values (critical!)
print('[2] Missing value analysis')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)
print(missing_df[missing_df['missing_count'] > 0])

print('\n[3] Target variable distribution')
print(df['survived'].value_counts())
print(f'Positive class rate: {df["survived"].mean():.3f}')

In [ ]:
# EDA CHECKLIST POINTS 4-8: Numerical and categorical analysis
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('EDA Checklist Points 4-8: Variable Distributions', fontsize=14, y=1.02)

# [4] Numerical distributions
df['age'].hist(ax=axes[0,0], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('[4] Age Distribution')
axes[0,0].set_xlabel('Age')

# [5] Target relationship with numerical
df.boxplot(column='age', by='survived', ax=axes[0,1])
axes[0,1].set_title('[5] Age vs Survived')
axes[0,1].set_xlabel('Survived')

# [6] Categorical variable counts
df['pclass'].value_counts().plot(kind='bar', ax=axes[0,2], color='steelblue')
axes[0,2].set_title('[6] Passenger Class Counts')
axes[0,2].set_xlabel('Class')

# [7] Categorical vs target
survival_by_class = df.groupby('pclass')['survived'].mean()
survival_by_class.plot(kind='bar', ax=axes[1,0], color='coral')
axes[1,0].set_title('[7] Survival Rate by Class')
axes[1,0].set_xlabel('Class')
axes[1,0].set_ylabel('Survival Rate')

# [8] Fare distribution (log scale)
df['fare'].hist(ax=axes[1,1], bins=50, color='green', edgecolor='white')
axes[1,1].set_title('[8] Fare Distribution')
axes[1,1].set_xlabel('Fare')
axes[1,1].set_yscale('log')

# Survival by sex
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[1,2], color='purple')
axes[1,2].set_title('[9] Survival Rate by Sex')
axes[1,2].set_xlabel('Sex')
axes[1,2].set_ylabel('Survival Rate')

plt.tight_layout()
plt.show()
print('Checklist points 4-9 visualized.')

In [ ]:
# EDA CHECKLIST POINTS 10-15
print('[10] Correlation matrix of numerical features')
numerical_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr_matrix = df[numerical_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# [10] Correlation heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=axes[0])
axes[0].set_title('[10] Feature Correlation Matrix')

# [11] Outlier detection
df.boxplot(column='fare', ax=axes[1])
axes[1].set_title('[11] Fare Outliers')

plt.tight_layout()
plt.show()

print('\n[12] Duplicate rows:', df.duplicated().sum())
print('[13] Constant features (zero variance):')
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].nunique() == 1:
        print(f'  - {col}')
print('None found.' if True else '')

print('\n[14] Cardinality of categorical features:')
for col in df.select_dtypes(include=['object', 'category']).columns:
    print(f'  {col}: {df[col].nunique()} unique values')

print('\n[15] Train/Test distribution alignment check:')
print('  (In a real competition, compare train vs test feature distributions.')
print('  Large differences = covariate shift = need domain adaptation.)')

## 1.2 Validation Strategy: The Most Important Thing in Kaggle

**The golden rule**: Your validation strategy MUST mirror the test set. Get this wrong and you will waste weeks chasing a leaderboard that doesn't reflect reality.

### When to Use Each Strategy

| Strategy | Use Case |
|---|---|
| KFold | Regression, balanced classification |
| StratifiedKFold | Imbalanced classification |
| GroupKFold | Data with groups (same user/store/etc) |
| TimeSeriesSplit | Time-series data (no future leakage) |
| Repeated KFold | When you have high variance, small dataset |

In [ ]:
# Validation strategy demonstration
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit

X_demo = np.random.randn(1000, 10)
y_demo = np.random.randint(0, 2, 1000)

print('=== VALIDATION STRATEGY COMPARISON ===')

# Standard KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(f'\nKFold: {kf.get_n_splits()} folds')
for i, (train_idx, val_idx) in enumerate(kf.split(X_demo)):
    print(f'  Fold {i+1}: train={len(train_idx)}, val={len(val_idx)}')

print()

# StratifiedKFold - maintains class ratio
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'StratifiedKFold: maintains class distribution across folds')
for i, (train_idx, val_idx) in enumerate(skf.split(X_demo, y_demo)):
    train_rate = y_demo[train_idx].mean()
    val_rate = y_demo[val_idx].mean()
    print(f'  Fold {i+1}: train_pos_rate={train_rate:.3f}, val_pos_rate={val_rate:.3f}')

In [ ]:
# Local CV vs Leaderboard score correlation
# This is a critical concept: if your CV score and LB score do not move together,
# your validation is broken.

print('=== LOCAL CV vs LEADERBOARD SCORE CORRELATION ===')
print()
print('GOOD correlation scenario (what you want):')
print('  Experiment 1: CV=0.820, LB=0.818  -> delta=0.002 (consistent)')
print('  Experiment 2: CV=0.831, LB=0.829  -> delta=0.002 (consistent)')
print('  Experiment 3: CV=0.845, LB=0.843  -> delta=0.002 (consistent)')
print()
print('BAD correlation scenario (your validation is broken):')
print('  Experiment 1: CV=0.820, LB=0.818')
print('  Experiment 2: CV=0.831, LB=0.811  -> DROPPED on LB despite CV improvement')
print('  Experiment 3: CV=0.845, LB=0.855  -> LB jumped without CV movement')
print()
print('Root causes of bad CV-LB correlation:')
print('  1. Data leakage in feature engineering')
print('  2. Wrong validation split (e.g., random split on time-series data)')
print('  3. Target leakage (using future data)')
print('  4. Public LB is too small a sample')
print('  5. Different preprocessing on train vs test')

In [ ]:
# OVERFITTING THE PUBLIC LEADERBOARD
# This is one of the most common expensive mistakes in Kaggle.

print('=== OVERFITTING THE PUBLIC LEADERBOARD ===')
print()
print('The Public LB is typically 25-50% of the test data.')
print('The Private LB (final ranking) is the remaining 50-75%.')
print()
print('How it happens:')
print('  - You make 50+ submissions, each time picking the one that improves LB')
print('  - You are essentially doing gradient descent on the PUBLIC LB')
print('  - Your model overfits to the 25-50% public test set')
print('  - On final reveal (private LB), you SHAKE DOWN significantly')
print()
print('Famous example: The "shake" in competitions like Santander')
print('where teams in top 10 on public LB fell to 100+ on private LB.')
print()
print('Prevention:')
print('  1. Trust your local CV above all')
print('  2. Set a budget: max 5 submissions/day')
print('  3. Only submit when CV improves meaningfully')
print('  4. Keep 2 final submissions: one high CV, one high LB (for safety)')

---
# SECTION 2: Feature Engineering Masterclass

## Squeezing Every Signal from the Titanic Dataset

The Titanic dataset is a classic benchmark. But most people stop at the obvious features. Here we will extract every meaningful signal available.

> **Interview Tip**: Feature engineering is where domain knowledge meets ML. When asked "how would you improve this model?", always lead with feature engineering before mentioning complex algorithms.

In [ ]:
# Load fresh Titanic data and prepare it
df_raw = sns.load_dataset('titanic')
print('Raw Titanic columns:', list(df_raw.columns))
print(f'Shape: {df_raw.shape}')
print()
print('Sample rows:')
print(df_raw[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch',
              'fare', 'embarked', 'class', 'who', 'deck']].head(10))

In [ ]:
# FEATURE 1: Extract title from name
# The seaborn Titanic dataset does not have the raw 'Name' column like the Kaggle version.
# We will simulate it by reconstructing typical title distributions.

# In the real Kaggle competition, you extract title from the Name column:
# df['title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
# Here we use the 'who' column as a proxy (adult male / adult female / child)

df = df_raw.copy()

# Map who to a title proxy
title_map = {'man': 'Mr', 'woman': 'Mrs', 'child': 'Master'}
df['title'] = df['who'].map(title_map)

# Show title distribution and survival rate
title_survival = df.groupby('title')['survived'].agg(['mean', 'count'])
title_survival.columns = ['survival_rate', 'count']
print('Title Survival Analysis:')
print(title_survival)

fig, ax = plt.subplots(figsize=(8, 4))
title_survival['survival_rate'].plot(kind='bar', ax=ax, color=['steelblue','coral','green'])
ax.set_title('Survival Rate by Title')
ax.set_ylabel('Survival Rate')
ax.set_xlabel('Title')
plt.tight_layout()
plt.show()

In [ ]:
# FEATURE 2: Family size
# sibsp = number of siblings/spouses aboard
# parch = number of parents/children aboard

df['family_size'] = df['sibsp'] + df['parch'] + 1  # +1 for the passenger themselves
df['is_alone'] = (df['family_size'] == 1).astype(int)

# Family size buckets - empirically found to matter more than raw count
df['family_group'] = pd.cut(
    df['family_size'],
    bins=[0, 1, 4, 7, 20],
    labels=['alone', 'small', 'medium', 'large']
)

family_survival = df.groupby('family_group')['survived'].mean()
print('Survival rate by family group:')
print(family_survival)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('family_size')['survived'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Survival Rate by Family Size')
axes[0].set_xlabel('Family Size')
axes[0].set_ylabel('Survival Rate')

family_survival.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Survival Rate by Family Group')
axes[1].set_xlabel('Family Group')
axes[1].set_ylabel('Survival Rate')

plt.tight_layout()
plt.show()

print(f'\nAlone passengers: {df["is_alone"].sum()} ({df["is_alone"].mean()*100:.1f}%)')
print(f'Alone survival rate: {df[df["is_alone"]==1]["survived"].mean():.3f}')
print(f'Non-alone survival rate: {df[df["is_alone"]==0]["survived"].mean():.3f}')

In [ ]:
# FEATURE 3: Cabin deck extraction
# The cabin column contains deck information (A, B, C, D, E, F, G, T)
# Most passengers have null cabin - that itself is informative!

df['has_cabin'] = df['deck'].notna().astype(int)
df['deck_clean'] = df['deck'].astype(str).str[0]
df.loc[df['deck'].isna(), 'deck_clean'] = 'Unknown'

print('Deck distribution:')
print(df['deck_clean'].value_counts())

deck_survival = df.groupby('deck_clean')['survived'].agg(['mean','count'])
deck_survival.columns = ['survival_rate', 'count']
print('\nDeck survival rates:')
print(deck_survival.sort_values('survival_rate', ascending=False))

# has_cabin is a powerful proxy for wealth/status
print(f'\nSurvival rate with cabin: {df[df["has_cabin"]==1]["survived"].mean():.3f}')
print(f'Survival rate without cabin: {df[df["has_cabin"]==0]["survived"].mean():.3f}')

In [ ]:
# FEATURE 4: Fare per person
# The fare column is the total fare for the entire group traveling together.
# Dividing by family size gives a better per-person wealth indicator.

df['fare_per_person'] = df['fare'] / df['family_size']

# Log transform fare (heavy right skew)
df['fare_log'] = np.log1p(df['fare'])
df['fare_per_person_log'] = np.log1p(df['fare_per_person'])

# Fill missing fare with median
df['fare_log'].fillna(df['fare_log'].median(), inplace=True)
df['fare_per_person_log'].fillna(df['fare_per_person_log'].median(), inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for survived_val, color, label in [(0, 'coral', 'Did not survive'), (1, 'steelblue', 'Survived')]:
    subset = df[df['survived'] == survived_val]['fare_log'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=label)
axes[0].set_title('Log Fare Distribution by Survival')
axes[0].set_xlabel('log(1 + Fare)')
axes[0].legend()

for survived_val, color, label in [(0, 'coral', 'Did not survive'), (1, 'steelblue', 'Survived')]:
    subset = df[df['survived'] == survived_val]['fare_per_person_log'].dropna()
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=label)
axes[1].set_title('Log Fare Per Person by Survival')
axes[1].set_xlabel('log(1 + Fare/FamilySize)')
axes[1].legend()

plt.tight_layout()
plt.show()

from scipy import stats
corr_fare, p_fare = stats.pointbiserialr(df['survived'], df['fare_log'].fillna(0))
corr_fpp, p_fpp = stats.pointbiserialr(
    df['survived'], df['fare_per_person_log'].fillna(0)
)
print(f'Correlation with survived: fare_log={corr_fare:.3f} (p={p_fare:.4f})')
print(f'Correlation with survived: fare_per_person_log={corr_fpp:.3f} (p={p_fpp:.4f})')

In [ ]:
# FEATURE 5: Age binning
# Continuous age is noisy. Bins capture non-linear relationships.

# Fill missing age with median by class and sex
df['age_filled'] = df.groupby(['pclass', 'sex'])['age'].transform(
    lambda x: x.fillna(x.median())
)

# Bin age into meaningful groups
df['age_bin'] = pd.cut(
    df['age_filled'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['child', 'teen', 'adult', 'middle_aged', 'senior']
)

age_survival = df.groupby('age_bin')['survived'].agg(['mean', 'count'])
age_survival.columns = ['survival_rate', 'count']
print('Survival rate by age bin:')
print(age_survival)

fig, ax = plt.subplots(figsize=(8, 4))
age_survival['survival_rate'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Survival Rate by Age Group')
ax.set_ylabel('Survival Rate')
ax.set_xlabel('Age Group')
for i, (rate, count) in enumerate(zip(age_survival['survival_rate'],
                                      age_survival['count'])):
    ax.text(i, rate + 0.01, f'n={count}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# FEATURE 6: Interaction features
# Some features are more powerful in combination than individually.

# Sex x Class interaction
df['sex_pclass'] = df['sex'].astype(str) + '_' + df['pclass'].astype(str)
interaction_survival = df.groupby('sex_pclass')['survived'].agg(['mean', 'count'])
interaction_survival.columns = ['survival_rate', 'count']
print('Sex x Class interaction survival rates:')
print(interaction_survival.sort_values('survival_rate', ascending=False))

# Age x Class interaction
df['is_child'] = (df['age_filled'] < 12).astype(int)
df['child_1st_class'] = ((df['is_child'] == 1) & (df['pclass'] == 1)).astype(int)
df['woman_1st_class'] = ((df['sex'] == 'female') & (df['pclass'] == 1)).astype(int)
df['man_3rd_class'] = ((df['sex'] == 'male') & (df['pclass'] == 3)).astype(int)

print('\nInteraction feature survival rates:')
for feat in ['child_1st_class', 'woman_1st_class', 'man_3rd_class']:
    pos = df[df[feat] == 1]['survived'].mean()
    print(f'  {feat}: {pos:.3f} ({(df[feat]==1).sum()} passengers)')

## 2.2 Target Encoding Done Correctly (with Cross-Validation)

Target encoding replaces a categorical variable with the mean target value for that category. But naive target encoding leaks information from the validation fold into the training fold, causing overfitting.

**The correct way**: Use out-of-fold target encoding within your CV loop.

In [ ]:
# TARGET ENCODING WITH CROSS-VALIDATION
# This prevents target leakage while still using target statistics.

def target_encode_cv(train_df, col, target, n_splits=5, smoothing=10):
    """
    Out-of-fold target encoding with smoothing.
    
    Smoothing formula: encoded = (count * category_mean + smoothing * global_mean)
                                  / (count + smoothing)
    
    This regularizes rare categories toward the global mean.
    """
    encoded = pd.Series(index=train_df.index, dtype=float)
    global_mean = train_df[target].mean()
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for train_idx, val_idx in skf.split(train_df, train_df[target]):
        train_fold = train_df.iloc[train_idx]
        
        # Calculate per-category statistics from training fold only
        stats = train_fold.groupby(col)[target].agg(['mean', 'count'])
        
        # Apply smoothing: rare categories blend toward global mean
        stats['smoothed'] = (
            (stats['count'] * stats['mean'] + smoothing * global_mean)
            / (stats['count'] + smoothing)
        )
        
        # Apply encoding to validation fold
        val_col = train_df.iloc[val_idx][col].map(stats['smoothed'])
        val_col.fillna(global_mean, inplace=True)
        encoded.iloc[val_idx] = val_col.values
    
    return encoded

# Apply target encoding to embarked column
df_enc = df.copy()
df_enc['embarked_target_enc'] = target_encode_cv(
    df_enc.dropna(subset=['embarked', 'survived']),
    col='embarked',
    target='survived'
)

# Compare naive vs CV target encoding
print('Naive target encoding (LEAKS information):')
naive_enc = df.groupby('embarked')['survived'].mean()
print(naive_enc)

print('\nCV-based target encoding (safe to use):')
cv_enc = df_enc.groupby('embarked')['embarked_target_enc'].mean().dropna()
print(cv_enc)

print('\nSmoothing effect on rare categories:')
print('  Rare category with only 3 examples will not be encoded as pure 100% survival.')
print('  Instead it blends toward the global mean, preventing overfitting.')

In [ ]:
# POLYNOMIAL FEATURES - When they help and when they hurt

print('=== POLYNOMIAL FEATURES: WHEN TO USE ===')
print()
print('GOOD use cases:')
print('  - Linear models (LR, Ridge, Lasso) that cannot capture interactions')
print('  - Small feature sets (< 20 features) - explosion in combinations')
print('  - When you have domain knowledge about multiplicative effects')
print('  - When your CV score plateaus and you want more expressive power')
print()
print('BAD use cases:')
print('  - Tree-based models (RF, GBM, XGBoost) - they already capture interactions')
print('  - Large feature sets - combinatorial explosion is computationally expensive')
print('  - High-dimensional sparse data')
print()

# Demonstrate with a small example
from sklearn.preprocessing import PolynomialFeatures

X_small = df[['fare_log', 'age_filled']].fillna(0).values[:100]
print(f'Original feature space: {X_small.shape}')

poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_poly2 = poly2.fit_transform(X_small)
print(f'Degree 2 polynomial features: {X_poly2.shape}')
print(f'New features: {poly2.get_feature_names_out(["fare_log", "age_filled"])}')

poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_poly3 = poly3.fit_transform(X_small)
print(f'\nDegree 3 polynomial features: {X_poly3.shape}')
print('  -> Combinatorial explosion: use with caution!')

---
# SECTION 3: Stacking and Blending

## The Power of Model Ensembles

Stacking is the most powerful ensemble technique used in virtually every Kaggle winning solution. The key insight: **different models make different errors**, and a meta-learner can learn which model to trust for which inputs.

### Stacking Architecture

```
Level 1 (Base Models):
  - Logistic Regression     --> OOF predictions
  - Random Forest           --> OOF predictions
  - Gradient Boosting       --> OOF predictions
  - SVM                     --> OOF predictions
  - KNN                     --> OOF predictions

                               |
                               v
                    [New Feature Matrix]
                    (5 columns = 5 models)
                               |
                               v
Level 2 (Meta-Learner):
  - Logistic Regression  --> Final prediction
```

In [ ]:
# Build the feature matrix for stacking

def build_titanic_features(df_input):
    """Build clean feature matrix from Titanic data."""
    df_f = df_input.copy()
    
    # Encode sex
    df_f['sex_enc'] = (df_f['sex'] == 'female').astype(int)
    
    # Embarked encoding
    df_f['embarked_enc'] = df_f['embarked'].map({'S': 0, 'C': 1, 'Q': 2}).fillna(0)
    
    # Family features
    df_f['family_size'] = df_f['sibsp'] + df_f['parch'] + 1
    df_f['is_alone'] = (df_f['family_size'] == 1).astype(int)
    
    # Age - fill missing
    df_f['age_filled'] = df_f.groupby(['pclass', 'sex'])['age'].transform(
        lambda x: x.fillna(x.median())
    )
    df_f['age_filled'].fillna(df_f['age_filled'].median(), inplace=True)
    df_f['is_child'] = (df_f['age_filled'] < 12).astype(int)
    
    # Fare
    df_f['fare_filled'] = df_f['fare'].fillna(df_f['fare'].median())
    df_f['fare_log'] = np.log1p(df_f['fare_filled'])
    df_f['fare_per_person'] = df_f['fare_log'] / df_f['family_size']
    
    # Has cabin
    df_f['has_cabin'] = df_f['deck'].notna().astype(int)
    
    # Interaction features
    df_f['woman_1st'] = ((df_f['sex'] == 'female') & (df_f['pclass'] == 1)).astype(int)
    df_f['man_3rd'] = ((df_f['sex'] == 'male') & (df_f['pclass'] == 3)).astype(int)
    
    feature_cols = [
        'pclass', 'sex_enc', 'age_filled', 'sibsp', 'parch',
        'fare_log', 'fare_per_person', 'embarked_enc',
        'family_size', 'is_alone', 'is_child', 'has_cabin',
        'woman_1st', 'man_3rd'
    ]
    
    return df_f[feature_cols]

# Prepare data
df_titanic = df_raw.dropna(subset=['survived']).copy()
X_titanic = build_titanic_features(df_titanic)
y_titanic = df_titanic['survived'].values

print(f'Feature matrix shape: {X_titanic.shape}')
print(f'Target shape: {y_titanic.shape}')
print(f'Positive class rate: {y_titanic.mean():.3f}')
print(f'\nFeatures: {list(X_titanic.columns)}')

In [ ]:
# LEVEL 1: Generate out-of-fold predictions for each base model

def get_oof_predictions(model, X, y, n_splits=5):
    """
    Generate out-of-fold predictions.
    
    This is the core of stacking: each sample is predicted ONLY
    by models trained on data that EXCLUDED that sample.
    This prevents leakage into the meta-learner.
    """
    oof_preds = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold = X.iloc[train_idx] if hasattr(X, 'iloc') else X[train_idx]
        X_val_fold = X.iloc[val_idx] if hasattr(X, 'iloc') else X[val_idx]
        y_train_fold = y[train_idx]
        
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_train_fold, y_train_fold)
        
        if hasattr(model_clone, 'predict_proba'):
            oof_preds[val_idx] = model_clone.predict_proba(X_val_fold)[:, 1]
        else:
            oof_preds[val_idx] = model_clone.decision_function(X_val_fold)
            # Normalize SVM scores to [0,1]
            min_v, max_v = oof_preds[val_idx].min(), oof_preds[val_idx].max()
            if max_v > min_v:
                oof_preds[val_idx] = (oof_preds[val_idx] - min_v) / (max_v - min_v)
    
    return oof_preds

# Scale features for models that need it
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_titanic)

# Define Level 1 models
base_models = {
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7)
}

print('Generating OOF predictions for Level 1 models...')
print('(This is the core of stacking - takes a moment)')
print()

In [ ]:
# Generate OOF predictions for all base models
oof_predictions = {}
model_scores = {}

for name, model in base_models.items():
    oof_preds = get_oof_predictions(model, X_scaled, y_titanic, n_splits=5)
    oof_predictions[name] = oof_preds
    auc = roc_auc_score(y_titanic, oof_preds)
    acc = accuracy_score(y_titanic, (oof_preds > 0.5).astype(int))
    model_scores[name] = {'AUC': auc, 'Accuracy': acc}
    print(f'{name:25s}: AUC={auc:.4f}, Accuracy={acc:.4f}')

print()
print('OOF prediction matrix built successfully.')
print('This becomes the NEW feature matrix for Level 2.')

In [ ]:
# LEVEL 2: Meta-learner on OOF predictions

# Build the stacking feature matrix
X_meta = np.column_stack(list(oof_predictions.values()))
print(f'Meta-learner input shape: {X_meta.shape}')
print(f'(Each column = OOF predictions from one base model)')
print()

# Fit the meta-learner
meta_learner = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
stacking_oof = get_oof_predictions(meta_learner, X_meta, y_titanic, n_splits=5)
stacking_auc = roc_auc_score(y_titanic, stacking_oof)
stacking_acc = accuracy_score(y_titanic, (stacking_oof > 0.5).astype(int))

print('=== STACKING RESULTS SUMMARY ===')
print()
print('Individual models:')
for name, scores in model_scores.items():
    print(f'  {name:25s}: AUC={scores["AUC"]:.4f}, Accuracy={scores["Accuracy"]:.4f}')

print(f'\nStacking (meta-learner):     AUC={stacking_auc:.4f}, Accuracy={stacking_acc:.4f}')
best_individual_auc = max(s['AUC'] for s in model_scores.values())
best_individual_acc = max(s['Accuracy'] for s in model_scores.values())
print(f'\nImprovement over best individual:')
print(f'  AUC: +{stacking_auc - best_individual_auc:.4f}')
print(f'  Accuracy: +{stacking_acc - best_individual_acc:.4f}')

In [ ]:
# Visualize: Stacking vs Individual Models
model_names = list(model_scores.keys()) + ['STACKING']
aucs = [model_scores[n]['AUC'] for n in model_scores] + [stacking_auc]
accs = [model_scores[n]['Accuracy'] for n in model_scores] + [stacking_acc]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue'] * len(model_scores) + ['gold']

axes[0].bar(range(len(model_names)), aucs, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(range(len(model_names)))
axes[0].set_xticklabels(model_names, rotation=30, ha='right')
axes[0].set_ylabel('AUC Score')
axes[0].set_title('AUC: Individual Models vs Stacking')
axes[0].set_ylim(min(aucs) - 0.02, max(aucs) + 0.02)
axes[0].axhline(stacking_auc, color='red', linestyle='--', linewidth=1, label='Stacking AUC')
axes[0].legend()

axes[1].bar(range(len(model_names)), accs, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xticks(range(len(model_names)))
axes[1].set_xticklabels(model_names, rotation=30, ha='right')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy: Individual Models vs Stacking')
axes[1].set_ylim(min(accs) - 0.02, max(accs) + 0.02)

plt.tight_layout()
plt.show()
print('Gold bar = stacking ensemble.')

In [ ]:
# BLENDING vs STACKING - When to use each

print('=== BLENDING vs STACKING ===')
print()
print('STACKING (recommended for serious competitions):')
print('  + Uses all training data via OOF predictions')
print('  + More data efficient - no holdout set needed')
print('  + Generally produces better results')
print('  - More complex to implement')
print('  - Slower (trains N_models x K_folds times)')
print()
print('BLENDING (simpler alternative):')
print('  + Much simpler to implement')
print('  + Faster to run')
print('  - Wastes 20-30% of training data for holdout')
print('  - Works well when you have lots of data')
print()
print('--- BLENDING IMPLEMENTATION ---')

from sklearn.model_selection import train_test_split

# Blending: split train into train_blend and holdout
X_train_b, X_holdout, y_train_b, y_holdout = train_test_split(
    X_scaled, y_titanic, test_size=0.25, random_state=42, stratify=y_titanic
)

blend_preds_holdout = []
blend_model_names = []

for name, model in base_models.items():
    m = model.__class__(**model.get_params())
    m.fit(X_train_b, y_train_b)
    if hasattr(m, 'predict_proba'):
        pred = m.predict_proba(X_holdout)[:, 1]
    else:
        pred = m.decision_function(X_holdout)
        pred = (pred - pred.min()) / (pred.max() - pred.min())
    blend_preds_holdout.append(pred)
    blend_model_names.append(name)

# Stack holdout predictions for meta-learner input
X_blend = np.column_stack(blend_preds_holdout)
meta_blend = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
meta_blend.fit(X_blend, y_holdout)

blend_auc = roc_auc_score(y_holdout, meta_blend.predict_proba(X_blend)[:,1])
print(f'\nBlending AUC on holdout: {blend_auc:.4f}')
print(f'Stacking OOF AUC:        {stacking_auc:.4f}')
print(f'\nNote: Blending AUC is in-sample on holdout - stacking estimate is more honest.')

---
# SECTION 4: Hyperparameter Tuning at Scale

## From Grid Search to Bayesian Optimization

Hyperparameter tuning is often the difference between a good model and a great one. But naive grid search is computationally wasteful. Here we cover the full spectrum of tuning approaches.

### The Efficiency Spectrum

```
Grid Search  <--(least efficient)----(most efficient)--> Bayesian Optimization
   |                    |                      |                    |
 Exhaustive         Random Sample          TPE/GP            Population
  (slow)            (better)               (smart)           (parallel)
```

In [ ]:
# GRIDSEARCHCV vs RANDOMIZEDSEARCHCV - Efficiency comparison

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import time

print('=== GRID SEARCH vs RANDOMIZED SEARCH ===')
print()

# Define parameter grids
grid_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

total_combinations = 1
for v in grid_params.values():
    total_combinations *= len(v)
print(f'Grid search combinations: {total_combinations}')
print(f'With 5-fold CV: {total_combinations * 5} model fits required')
print()

# Random search with same parameters but fewer iterations
random_iterations = 20
print(f'Random search iterations: {random_iterations}')
print(f'With 5-fold CV: {random_iterations * 5} model fits required')
print(f'Coverage: {random_iterations/total_combinations*100:.1f}% of parameter space')
print()

# Demonstrate RandomizedSearchCV (faster)
from scipy.stats import randint, uniform

random_param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 15),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

rf = RandomForestClassifier(random_state=42)
rs = RandomizedSearchCV(
    rf, random_param_dist, n_iter=20, cv=3,
    scoring='roc_auc', n_jobs=-1, random_state=42, verbose=0
)

start = time.time()
rs.fit(X_scaled, y_titanic)
elapsed = time.time() - start

print(f'RandomizedSearchCV complete in {elapsed:.1f}s')
print(f'Best params: {rs.best_params_}')
print(f'Best CV AUC: {rs.best_score_:.4f}')

In [ ]:
# BAYESIAN OPTIMIZATION WITH OPTUNA
# Much smarter than random search - learns from previous trials

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
    print('Optuna is available. Running Bayesian optimization...')
except ImportError:
    OPTUNA_AVAILABLE = False
    print('Optuna not installed. Showing conceptual implementation.')
    print('Install with: pip install optuna')

if OPTUNA_AVAILABLE:
    def objective(trial):
        """Optuna objective function for Random Forest tuning."""
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        }
        
        model = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
        
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_scaled, y_titanic,
                                  cv=skf, scoring='roc_auc', n_jobs=-1)
        return scores.mean()
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, show_progress_bar=False)
    
    print(f'\nBayesian optimization complete (30 trials).')
    print(f'Best AUC: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')
    print()
    
    # Compare efficiency
    print('Comparison:')
    print(f'  RandomizedSearchCV (20 iter): AUC={rs.best_score_:.4f}')
    print(f'  Optuna Bayesian (30 trials):  AUC={study.best_value:.4f}')
    print()
    print('Bayesian optimization is smarter because each trial informs the next.')
else:
    print()
    print('Conceptual code for Optuna Bayesian optimization:')
    print('''
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
    }
    model = GradientBoostingClassifier(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)
print(study.best_params)
    ''')

In [ ]:
# EARLY STOPPING IN GRADIENT BOOSTING
# Instead of tuning n_estimators, use early stopping to find it automatically.

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_titanic, test_size=0.2, random_state=42, stratify=y_titanic
)

# Track validation scores at each iteration
gbm = GradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, random_state=42
)
gbm.fit(X_train, y_train)

# Staged predictions (one per tree)
train_scores = []
val_scores = []
for i, pred in enumerate(gbm.staged_predict_proba(X_train)):
    train_scores.append(roc_auc_score(y_train, pred[:, 1]))
for i, pred in enumerate(gbm.staged_predict_proba(X_val)):
    val_scores.append(roc_auc_score(y_val, pred[:, 1]))

best_iter = np.argmax(val_scores)
print(f'Best iteration (early stopping): {best_iter + 1}')
print(f'Best validation AUC: {val_scores[best_iter]:.4f}')
print(f'Training AUC at best iter: {train_scores[best_iter]:.4f}')
print(f'Final iteration (500): val AUC={val_scores[-1]:.4f} (overfit!)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(train_scores)+1), train_scores, label='Train AUC', color='steelblue')
ax.plot(range(1, len(val_scores)+1), val_scores, label='Val AUC', color='coral')
ax.axvline(best_iter + 1, color='green', linestyle='--',
           label=f'Early stop @ iter {best_iter+1}')
ax.set_xlabel('Number of Trees')
ax.set_ylabel('AUC Score')
ax.set_title('Early Stopping: Finding the Optimal Number of Trees')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# LEARNING CURVES - Diagnose over/underfitting

from sklearn.model_selection import learning_curve

def plot_learning_curves(model, X, y, title, cv=5):
    """Plot training and validation learning curves."""
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=cv, scoring='roc_auc',
        n_jobs=-1, random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
    plt.plot(train_sizes, train_mean, 'b-o', label='Training AUC', markersize=4)
    plt.plot(train_sizes, val_mean, 'o-', color='orange', label='Validation AUC', markersize=4)
    plt.xlabel('Training Set Size')
    plt.ylabel('AUC Score')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plt.subplot(1, 2, 1)
plot_learning_curves(
    LogisticRegression(max_iter=1000), X_scaled, y_titanic,
    'Learning Curves: Logistic Regression (Underfitting?)'
)

plt.subplot(1, 2, 2)
plot_learning_curves(
    RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42),
    X_scaled, y_titanic,
    'Learning Curves: Random Forest (Overfitting?)'
)

plt.tight_layout()
plt.show()

print('Interpretation guide:')
print('  HIGH train, LOW val -> OVERFITTING -> regularize, add data, reduce complexity')
print('  LOW train, LOW val  -> UNDERFITTING -> add features, increase complexity')
print('  Both converge LOW   -> NEED MORE DATA or better features')
print('  Both converge HIGH  -> GOOD FIT')

In [ ]:
# FEATURE SELECTION TO REDUCE SEARCH SPACE

from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.inspection import permutation_importance

print('=== FEATURE SELECTION STRATEGIES ===')
print()

# Method 1: Tree-based feature importance
rf_full = RandomForestClassifier(n_estimators=200, random_state=42)
rf_full.fit(X_scaled, y_titanic)

importances = pd.Series(
    rf_full.feature_importances_,
    index=X_titanic.columns
).sort_values(ascending=False)

print('Method 1: Random Forest Feature Importances:')
for feat, imp in importances.items():
    bar = '#' * int(imp * 100)
    print(f'  {feat:20s}: {imp:.4f} {bar}')

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importances (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

# Method 2: Recursive Feature Elimination
rfe = RFE(
    RandomForestClassifier(n_estimators=50, random_state=42),
    n_features_to_select=8
)
rfe.fit(X_scaled, y_titanic)
selected_features = X_titanic.columns[rfe.support_].tolist()
print(f'\nMethod 2: RFE selected features (top 8): {selected_features}')

---
# SECTION 5: Real Competition Techniques

## The Advanced Arsenal That Winners Use

These techniques appear in nearly every top Kaggle solution. Understanding them is essential for senior ML interviews.

> **Interview Tip**: Mentioning pseudo-labeling and knowing its pitfalls signals production ML experience.

In [ ]:
# TECHNIQUE 1: PSEUDO-LABELING (Semi-supervised learning)

print('=== PSEUDO-LABELING ===')
print()
print('Concept: Use high-confidence test predictions as additional training data.')
print()
print('Algorithm:')
print('  1. Train model on labeled data')
print('  2. Predict on unlabeled test data')
print('  3. Keep predictions with confidence > threshold (e.g., 0.95)')
print('  4. Add these high-confidence pseudo-labeled samples to training data')
print('  5. Retrain on original + pseudo-labeled data')
print('  6. Optionally iterate (but watch for error propagation)')
print()

def pseudo_label_iteration(X_train, y_train, X_test, model, threshold=0.95, max_iter=3):
    """
    Iterative pseudo-labeling for semi-supervised learning.
    
    Parameters:
        X_train: labeled training features
        y_train: training labels
        X_test: unlabeled test features (pseudo-labels generated for these)
        model: base classifier
        threshold: confidence threshold for accepting pseudo-labels
        max_iter: maximum iterations
    
    Returns:
        Final trained model and pseudo-label history.
    """
    history = []
    X_aug = X_train.copy()
    y_aug = y_train.copy()
    
    for iteration in range(max_iter):
        model_iter = model.__class__(**model.get_params())
        model_iter.fit(X_aug, y_aug)
        
        test_proba = model_iter.predict_proba(X_test)[:, 1]
        
        # Select high-confidence predictions
        high_conf_mask = (test_proba > threshold) | (test_proba < (1 - threshold))
        pseudo_labels = (test_proba > 0.5).astype(int)
        
        n_pseudo = high_conf_mask.sum()
        
        train_auc = roc_auc_score(y_train, model_iter.predict_proba(X_train)[:,1])
        history.append({'iteration': iteration + 1,
                        'n_pseudo': n_pseudo,
                        'train_size': len(X_aug),
                        'train_auc': train_auc})
        
        print(f'  Iteration {iteration+1}: {n_pseudo} pseudo-labels added '
              f'(confidence > {threshold}), train_size={len(X_aug)}, '
              f'train_AUC={train_auc:.4f}')
        
        if n_pseudo == 0:
            print('  No new pseudo-labels. Stopping.')
            break
        
        # Add pseudo-labeled samples
        X_aug = np.vstack([X_aug, X_test[high_conf_mask]])
        y_aug = np.concatenate([y_aug, pseudo_labels[high_conf_mask]])
    
    return model_iter, history

# Simulate train/test split for demonstration
X_pl_train, X_pl_test, y_pl_train, y_pl_test = train_test_split(
    X_scaled, y_titanic, test_size=0.3, random_state=42, stratify=y_titanic
)

print('Running pseudo-labeling simulation:')
base_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model, pl_history = pseudo_label_iteration(
    X_pl_train, y_pl_train, X_pl_test, base_model, threshold=0.90
)

# Compare baseline vs pseudo-labeled model
baseline = RandomForestClassifier(n_estimators=100, random_state=42)
baseline.fit(X_pl_train, y_pl_train)
baseline_auc = roc_auc_score(y_pl_test, baseline.predict_proba(X_pl_test)[:,1])
final_auc = roc_auc_score(y_pl_test, final_model.predict_proba(X_pl_test)[:,1])

print(f'\nBaseline AUC (no pseudo-labeling): {baseline_auc:.4f}')
print(f'Final AUC (with pseudo-labeling):  {final_auc:.4f}')

In [ ]:
# TECHNIQUE 2: TEST TIME AUGMENTATION (TTA)

print('=== TEST TIME AUGMENTATION (TTA) ===')
print()
print('Original use: Computer Vision (flip, rotate, crop the test image multiple times')
print('and average predictions). Also applies to tabular data.')
print()
print('Tabular TTA strategies:')
print('  1. Feature perturbation: add small noise to numerical features, average predictions')
print('  2. Feature dropout: randomly zero out features, average predictions')
print('  3. Feature ordering: for models sensitive to order, try different orderings')
print()

def tta_predict(model, X_test, n_augments=10, noise_std=0.01):
    """
    Test Time Augmentation for tabular data.
    
    Adds small Gaussian noise to test features and averages predictions
    across augmentations. Reduces variance in predictions.
    """
    all_preds = []
    
    # Original prediction
    all_preds.append(model.predict_proba(X_test)[:, 1])
    
    # Augmented predictions
    for _ in range(n_augments - 1):
        X_noisy = X_test + np.random.randn(*X_test.shape) * noise_std
        all_preds.append(model.predict_proba(X_noisy)[:, 1])
    
    return np.mean(all_preds, axis=0)

# Compare regular prediction vs TTA
rf_tta = RandomForestClassifier(n_estimators=100, random_state=42)
rf_tta.fit(X_pl_train, y_pl_train)

regular_preds = rf_tta.predict_proba(X_pl_test)[:, 1]
tta_preds = tta_predict(rf_tta, X_pl_test, n_augments=20, noise_std=0.05)

regular_auc = roc_auc_score(y_pl_test, regular_preds)
tta_auc = roc_auc_score(y_pl_test, tta_preds)

print(f'Regular prediction AUC: {regular_auc:.4f}')
print(f'TTA prediction AUC:     {tta_auc:.4f}')
print()
print('TTA benefit is larger when:')
print('  - Model has high prediction variance')
print('  - Test samples have measurement noise')
print('  - You are using deep learning (larger benefit than tree models)')

In [ ]:
# TECHNIQUE 3: RANK AVERAGING vs WEIGHTED AVERAGING

print('=== RANK AVERAGING vs WEIGHTED AVERAGING ===')
print()

# Generate predictions from multiple models
pred_lr = LogisticRegression(max_iter=1000, random_state=42).fit(
    X_pl_train, y_pl_train).predict_proba(X_pl_test)[:, 1]
pred_rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(
    X_pl_train, y_pl_train).predict_proba(X_pl_test)[:, 1]
pred_gb = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(
    X_pl_train, y_pl_train).predict_proba(X_pl_test)[:, 1]

print('Individual model AUCs:')
for name, pred in [('LR', pred_lr), ('RF', pred_rf), ('GB', pred_gb)]:
    print(f'  {name}: {roc_auc_score(y_pl_test, pred):.4f}')

# Method 1: Simple average
simple_avg = (pred_lr + pred_rf + pred_gb) / 3
simple_auc = roc_auc_score(y_pl_test, simple_avg)
print(f'\nSimple average AUC: {simple_auc:.4f}')

# Method 2: Weighted average (weight by individual AUC)
auc_lr = roc_auc_score(y_pl_test, pred_lr)
auc_rf = roc_auc_score(y_pl_test, pred_rf)
auc_gb = roc_auc_score(y_pl_test, pred_gb)
total_auc = auc_lr + auc_rf + auc_gb

weighted_avg = (pred_lr * auc_lr + pred_rf * auc_rf + pred_gb * auc_gb) / total_auc
weighted_auc = roc_auc_score(y_pl_test, weighted_avg)
print(f'Weighted average AUC: {weighted_auc:.4f}')

# Method 3: RANK averaging - more robust
# Converts predictions to ranks first, then averages
from scipy.stats import rankdata

rank_lr = rankdata(pred_lr) / len(pred_lr)
rank_rf = rankdata(pred_rf) / len(pred_rf)
rank_gb = rankdata(pred_gb) / len(pred_gb)

rank_avg = (rank_lr + rank_rf + rank_gb) / 3
rank_auc = roc_auc_score(y_pl_test, rank_avg)
print(f'Rank average AUC:     {rank_auc:.4f}')

print()
print('Why rank averaging is often better:')
print('  - Not affected by scale differences between models')
print('  - More robust to outlier predictions')
print('  - Especially useful when mixing probability and decision function outputs')
print('  - AUC is rank-based, so rank averaging aligns with the metric')

In [ ]:
# TECHNIQUE 4: OUT-OF-FOLD PREDICTIONS - Theory and Practice

print('=== OUT-OF-FOLD (OOF) PREDICTIONS ===')
print()
print('OOF predictions are the backbone of stacking.')
print()
print('Why OOF is crucial for stacking:')
print('  - If you use in-sample predictions, the meta-learner sees "perfect" base model outputs')
print('  - The base models memorize training data -> meta-learner overfits to memorized patterns')
print('  - OOF ensures every prediction in the meta-training set is out-of-sample')
print()
print('Visualization of OOF process:')
print()
print('Fold 1: [TRAIN][TRAIN][TRAIN][TRAIN] [VAL ] -> predict VAL with model trained on TRAIN')
print('Fold 2: [TRAIN][TRAIN][TRAIN][VAL ][TRAIN] -> predict VAL with model trained on TRAIN')
print('Fold 3: [TRAIN][TRAIN][VAL ][TRAIN][TRAIN] -> predict VAL with model trained on TRAIN')
print('Fold 4: [TRAIN][VAL ][TRAIN][TRAIN][TRAIN] -> predict VAL with model trained on TRAIN')
print('Fold 5: [VAL ][TRAIN][TRAIN][TRAIN][TRAIN] -> predict VAL with model trained on TRAIN')
print()
print('Result: Every sample has exactly ONE out-of-sample prediction.')
print('Concatenate all -> OOF predictions for full training set.')
print()

# Verify our OOF implementation is correct
assert len(oof_predictions['RandomForest']) == len(y_titanic), \
    'OOF predictions must cover all training samples'
print(f'OOF coverage verified: {len(oof_predictions["RandomForest"])} predictions '
      f'for {len(y_titanic)} samples.')
print()
print('Each sample predicted exactly once (by model trained without it).')

In [ ]:
# TECHNIQUE 5: THE SHAKE - Public vs Private Leaderboard

print('=== THE SHAKE: WHY PUBLIC LB DIFFERS FROM PRIVATE LB ===')
print()
print('Competition leaderboard structure:')
print('  - Public LB:  ~30% of test data (shown during competition)')
print('  - Private LB: ~70% of test data (revealed at competition end)')
print()
print('Sources of shake:')
print()
print('1. RANDOM VARIANCE')
print('   Public LB sample is too small to be statistically reliable.')
print('   Solution: Trust your CV score, not the LB.')
print()
print('2. OVERFITTING TO PUBLIC LB')
print('   Multiple submissions -> you tune to the 30% public sample.')
print('   Solution: Limit daily submissions. Treat each as precious.')
print()
print('3. DATA DISTRIBUTION SHIFT')
print('   Public and private test sets may have different distributions.')
print('   Solution: Build robust models that do not overfit to specific patterns.')
print()
print('4. TIME-BASED SPLITS')
print('   Public LB is older data, private LB is newer data.')
print('   Your model may not generalize to the future.')
print('   Solution: Use time-aware validation.')
print()

# Simulate the shake statistically
np.random.seed(42)
true_model_auc = 0.85  # True model performance

public_lb_sizes = [50, 100, 200, 500, 1000, 5000]
shake_magnitudes = []

for n in public_lb_sizes:
    # Simulate 1000 public LB samples
    scores = []
    for _ in range(1000):
        labels = np.random.binomial(1, 0.4, n)
        if labels.sum() == 0 or labels.sum() == n:
            continue
        preds = np.clip(
            np.random.normal(true_model_auc, 0.1, n),
            0, 1
        )
        scores.append(roc_auc_score(labels, preds))
    shake_magnitudes.append(np.std(scores))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(public_lb_sizes, shake_magnitudes, 'o-', color='coral')
ax.set_xlabel('Public LB Sample Size')
ax.set_ylabel('Standard Deviation of LB Score')
ax.set_title('LB Score Variance vs Public LB Size')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Smaller public LB = more shake = less reliable ranking.')

---
# SECTION 6: End-to-End Kaggle Workflow

## Full Pipeline: Load -> EDA -> Features -> Model -> Ensemble

This section puts everything together in a realistic competition workflow using the California Housing dataset (substitute for the House Prices competition).

**Goal**: Predict median house values and beat the baseline by a significant margin.

> **Interview Tip**: Being able to describe an end-to-end ML pipeline from data loading to final ensemble is a core senior data scientist skill. Practice this until it is second nature.

In [ ]:
# STEP 1: Load California Housing dataset
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error

cal_housing = fetch_california_housing()
df_cal = pd.DataFrame(cal_housing.data, columns=cal_housing.feature_names)
df_cal['target'] = cal_housing.target  # Median house value in $100k

print('=== CALIFORNIA HOUSING DATASET ===')
print(f'Shape: {df_cal.shape}')
print(f'Target: {cal_housing.target_names}')
print(f'\nFeatures:')
for col, desc in zip(cal_housing.feature_names,
                      ['Median income', 'House age', 'Avg rooms',
                       'Avg bedrooms', 'Population', 'Avg occupancy',
                       'Latitude', 'Longitude']):
    print(f'  {col:12s}: {desc}')
print(f'\nTarget stats:')
print(f'  Mean: ${df_cal["target"].mean()*100:.0f}k')
print(f'  Std:  ${df_cal["target"].std()*100:.0f}k')
print(f'  Min:  ${df_cal["target"].min()*100:.0f}k')
print(f'  Max:  ${df_cal["target"].max()*100:.0f}k')

In [ ]:
# STEP 2: Quick EDA (competition-paced)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('California Housing EDA', fontsize=14)

for i, col in enumerate(cal_housing.feature_names):
    ax = axes[i // 4][i % 4]
    ax.scatter(df_cal[col], df_cal['target'], alpha=0.1, s=1, color='steelblue')
    corr = df_cal[col].corr(df_cal['target'])
    ax.set_xlabel(col)
    ax.set_ylabel('Target')
    ax.set_title(f'{col} (r={corr:.2f})')

plt.tight_layout()
plt.show()

print('Key observations:')
print('  MedInc has strong positive correlation with target (r=0.69)')
print('  Latitude/Longitude have geographic clusters (LA vs SF)')
print('  AveOccup has extreme outliers (need capping or log transform)')

In [ ]:
# STEP 3: Feature engineering for California Housing

df_feat = df_cal.copy()

# Geographic features
df_feat['geo_cluster'] = (
    (df_feat['Latitude'] > 37.5).astype(int) * 2 +
    (df_feat['Longitude'] > -119).astype(int)
)

# Interaction features
df_feat['rooms_per_person'] = df_feat['AveRooms'] / (df_feat['AveOccup'] + 1e-6)
df_feat['bedrooms_ratio'] = df_feat['AveBedrms'] / (df_feat['AveRooms'] + 1e-6)
df_feat['income_rooms'] = df_feat['MedInc'] * df_feat['AveRooms']
df_feat['pop_density'] = df_feat['Population'] / (df_feat['AveOccup'] + 1e-6)

# Log transform skewed features
df_feat['log_population'] = np.log1p(df_feat['Population'])
df_feat['log_ave_occup'] = np.log1p(df_feat['AveOccup'])

# Clip extreme outliers (top 1%)
for col in ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']:
    upper = df_feat[col].quantile(0.99)
    df_feat[col + '_capped'] = np.clip(df_feat[col], 0, upper)

# Distance to LA and SF (geographic anchors)
df_feat['dist_la'] = np.sqrt(
    (df_feat['Latitude'] - 34.05)**2 + (df_feat['Longitude'] - (-118.24))**2
)
df_feat['dist_sf'] = np.sqrt(
    (df_feat['Latitude'] - 37.77)**2 + (df_feat['Longitude'] - (-122.42))**2
)

print('Feature engineering complete.')
print(f'Original features: {len(cal_housing.feature_names)}')
print(f'Total features after engineering: {df_feat.drop("target", axis=1).shape[1]}')
print(f'\nNew features added:')
new_feats = [c for c in df_feat.columns
             if c not in cal_housing.feature_names and c != 'target']
for f in new_feats:
    print(f'  + {f}')

In [ ]:
# STEP 4: Build feature matrix and establish baseline

feature_cols = [c for c in df_feat.columns if c != 'target']
X_cal = df_feat[feature_cols].values
y_cal = df_feat['target'].values

X_cal_train, X_cal_test, y_cal_train, y_cal_test = train_test_split(
    X_cal, y_cal, test_size=0.2, random_state=42
)

scaler_cal = StandardScaler()
X_cal_train_sc = scaler_cal.fit_transform(X_cal_train)
X_cal_test_sc = scaler_cal.transform(X_cal_test)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# BASELINE: Predict the mean
baseline_pred = np.full(len(y_cal_test), y_cal_train.mean())
baseline_rmse = rmse(y_cal_test, baseline_pred)
print(f'BASELINE (predict mean): RMSE = {baseline_rmse:.4f}')
print(f'  (${baseline_rmse * 100:.0f}k average error)')
print()

# Simple Ridge regression
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_cal_train_sc, y_cal_train)
ridge_pred = ridge.predict(X_cal_test_sc)
ridge_rmse = rmse(y_cal_test, ridge_pred)
print(f'Ridge Regression: RMSE = {ridge_rmse:.4f} ({(1-ridge_rmse/baseline_rmse)*100:.1f}% better than baseline)')

In [ ]:
# STEP 5: Full model comparison

from sklearn.ensemble import ExtraTreesRegressor

models_reg = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42
    ),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=200, n_jobs=-1, random_state=42),
}

results_reg = {}
print('Model Comparison (Train -> Test RMSE):')
print(f'{"Model":20s} {"Train RMSE":12s} {"Test RMSE":12s} {"vs Baseline":12s}')
print('-' * 58)

for name, model in models_reg.items():
    X_train_input = X_cal_train_sc if name == 'Ridge' else X_cal_train
    X_test_input = X_cal_test_sc if name == 'Ridge' else X_cal_test
    
    model.fit(X_train_input, y_cal_train)
    train_pred = model.predict(X_train_input)
    test_pred = model.predict(X_test_input)
    
    train_rmse = rmse(y_cal_train, train_pred)
    test_rmse_val = rmse(y_cal_test, test_pred)
    improvement = (1 - test_rmse_val / baseline_rmse) * 100
    
    results_reg[name] = {
        'model': model,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse_val,
        'test_pred': test_pred
    }
    print(f'{name:20s} {train_rmse:12.4f} {test_rmse_val:12.4f} {improvement:+.1f}%')

print(f'\n{"BASELINE (mean)":20s} {"N/A":12s} {baseline_rmse:12.4f} {0:+.1f}%')

In [ ]:
# STEP 6: Build regression stacking ensemble

def get_oof_predictions_regression(model, X, y, n_splits=5):
    """OOF predictions for regression models."""
    oof_preds = np.zeros(len(y))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr = X[train_idx]
        X_vl = X[val_idx]
        y_tr = y[train_idx]
        
        m = model.__class__(**model.get_params())
        m.fit(X_tr, y_tr)
        oof_preds[val_idx] = m.predict(X_vl)
    
    return oof_preds

print('Building regression stacking ensemble...')
oof_reg = {}

for name, info in results_reg.items():
    X_input = X_cal_train_sc if name == 'Ridge' else X_cal_train
    oof = get_oof_predictions_regression(info['model'], X_input, y_cal_train)
    oof_reg[name] = oof
    oof_rmse = rmse(y_cal_train, oof)
    print(f'  {name}: OOF RMSE = {oof_rmse:.4f}')

print()

# Build meta-learner input
X_meta_reg = np.column_stack(list(oof_reg.values()))
X_meta_test = np.column_stack([info['test_pred'] for info in results_reg.values()])

# Meta-learner: Ridge regression on OOF predictions
meta_ridge = Ridge(alpha=0.01)
meta_ridge.fit(X_meta_reg, y_cal_train)
meta_pred = meta_ridge.predict(X_meta_test)
meta_rmse = rmse(y_cal_test, meta_pred)

best_individual_rmse = min(info['test_rmse'] for info in results_reg.values())
best_individual_name = min(results_reg, key=lambda k: results_reg[k]['test_rmse'])

print(f'Best individual model ({best_individual_name}): RMSE = {best_individual_rmse:.4f}')
print(f'Stacking ensemble:                              RMSE = {meta_rmse:.4f}')
print(f'Improvement from stacking: {(best_individual_rmse - meta_rmse) / best_individual_rmse * 100:.2f}%')
print(f'Improvement over baseline: {(1 - meta_rmse/baseline_rmse)*100:.1f}%')

In [ ]:
# Final visualization: Complete workflow results

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: RMSE comparison
all_names = list(results_reg.keys()) + ['STACKING', 'BASELINE']
all_rmses = [results_reg[n]['test_rmse'] for n in results_reg] + [meta_rmse, baseline_rmse]
colors_viz = ['steelblue'] * len(results_reg) + ['gold', 'coral']

bars = axes[0].bar(range(len(all_names)), all_rmses, color=colors_viz, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(range(len(all_names)))
axes[0].set_xticklabels(all_names, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('RMSE (lower is better)')
axes[0].set_title('Model Comparison: RMSE')
for bar, val in zip(bars, all_rmses):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                  f'{val:.3f}', ha='center', va='bottom', fontsize=8)

# Plot 2: Actual vs predicted for stacking
axes[1].scatter(y_cal_test, meta_pred, alpha=0.2, s=5, color='steelblue')
axes[1].plot([y_cal_test.min(), y_cal_test.max()],
              [y_cal_test.min(), y_cal_test.max()], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Value ($100k)')
axes[1].set_ylabel('Predicted Value ($100k)')
axes[1].set_title('Stacking: Actual vs Predicted')
corr = np.corrcoef(y_cal_test, meta_pred)[0, 1]
axes[1].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[1].transAxes,
              fontsize=12, verticalalignment='top')

# Plot 3: Residuals
residuals = y_cal_test - meta_pred
axes[2].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residual ($100k)')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Residuals (std={residuals.std():.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# STEP 7: Meta-learner weights reveal model strengths

print('=== META-LEARNER ANALYSIS ===')
print()
print('Meta-learner coefficients (Ridge regression on OOF predictions):')
for name, coef in zip(results_reg.keys(), meta_ridge.coef_):
    bar = '#' * int(abs(coef) * 20)
    print(f'  {name:20s}: {coef:+.4f}  {bar}')

print()
print('Interpretation:')
print('  Positive coef = model adds value')
print('  Large coef = model is most trusted by meta-learner')
print('  Small/negative coef = model is redundant or slightly harmful')
print()
print('Feature importances from best individual model (GradientBoosting):')
gb_model = results_reg['GradientBoosting']['model']
feat_imp = pd.Series(gb_model.feature_importances_, index=feature_cols)
feat_imp_top = feat_imp.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 4))
feat_imp_top.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 10 Feature Importances (Gradient Boosting)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

---
# Summary: The Complete Kaggle Playbook

## What We Covered

| Section | Key Takeaway |
|---|---|
| Competition Framework | Build validation first. Trust CV over LB. Do not overfit public LB. |
| Feature Engineering | More time here = better results. Interactions, binning, target encoding. |
| Stacking | OOF predictions are mandatory. Meta-learner learns model strengths. |
| Hyperparameter Tuning | Bayesian > Random > Grid. Use early stopping. Learning curves diagnose fit. |
| Real Techniques | Pseudo-labeling, TTA, rank averaging, OOF, shake awareness. |
| End-to-End Workflow | Load -> EDA -> Features -> Baseline -> Models -> Ensemble -> Analyze. |

## The Grandmaster Mindset

1. **Understand the metric deeply** before writing any code
2. **Validation is sacred** - get it wrong and everything else is wasted effort
3. **Feature engineering beats model complexity** at nearly every Kaggle competition
4. **Ensembles almost always win** - diversity matters more than individual model quality
5. **Track every experiment** - you cannot improve what you do not measure
6. **Simplicity over complexity** when CV improvement is marginal

## Interview Answer Template

When asked "How would you approach a new ML competition/problem?":

```
1. Understand the problem and evaluation metric
2. Analyze training data distribution and missing values
3. Set up a robust validation framework that mirrors the test set
4. Build a minimal baseline to establish a reference point
5. Feature engineering: domain-driven + systematic (interactions, transforms, encoding)
6. Train diverse base models and compare via cross-validation
7. Ensemble: stacking with OOF predictions or rank averaging
8. Hyperparameter tune the most promising models with Bayesian optimization
9. Post-processing: pseudo-labeling if semi-supervised signal exists
10. Final selection: trust local CV, not public leaderboard
```

In [ ]:
# FINAL SUMMARY: Score comparison table

print('=' * 60)
print('COMPLETE RESULTS SUMMARY')
print('=' * 60)
print()
print(f'Dataset: California Housing ({len(df_cal):,} samples)')
print(f'Task: Regression (predict median house value)')
print(f'Metric: RMSE (lower is better)')
print()
print(f'{"Model":30s} {"RMSE":10s} {"vs Baseline":12s}')
print('-' * 54)

all_results = [
    ('Baseline (predict mean)', baseline_rmse, 0.0),
]
for name, info in results_reg.items():
    improvement = (1 - info['test_rmse'] / baseline_rmse) * 100
    all_results.append((name, info['test_rmse'], improvement))
improvement_stack = (1 - meta_rmse / baseline_rmse) * 100
all_results.append(('Stacking Ensemble', meta_rmse, improvement_stack))

for model_name, rmse_val, improvement in sorted(all_results, key=lambda x: x[1], reverse=True):
    tag = ' <-- WINNER' if model_name == 'Stacking Ensemble' else ''
    sign = '+' if improvement > 0 else ''
    print(f'{model_name:30s} {rmse_val:10.4f} {sign}{improvement:+.1f}%{tag}')

print()
print(f'Stacking beats best individual model by: '
      f'{(best_individual_rmse - meta_rmse) / best_individual_rmse * 100:.2f}%')
print(f'Stacking beats baseline by: {improvement_stack:.1f}%')
print()
print('This is consistent with real Kaggle competition results.')
print('Ensembling + Feature Engineering typically beats a single model by 5-15%.')

In [ ]:
# BONUS: Checklist for competition submission day

print('=== COMPETITION DAY CHECKLIST ===')
print()
checklist = [
    ('Validation', [
        'Local CV correlates with LB (r > 0.9)',
        'No data leakage in features',
        'Same preprocessing on train and test',
        'OOF predictions used for stacking (not in-sample)',
    ]),
    ('Features', [
        'Missing values handled consistently',
        'Target encoding uses CV (not raw statistics)',
        'No future leakage (time series)',
        'Feature distributions checked train vs test',
    ]),
    ('Models', [
        'Diverse base models (different algorithms)',
        'Base models tuned individually',
        'Meta-learner is simple (no overfitting)',
        'Final model selected by CV, not LB',
    ]),
    ('Submissions', [
        'Two final submissions chosen (best CV + best LB)',
        'Submission file format matches sample submission',
        'No NaN values in submission',
        'Submission within expected range',
    ]),
]

for category, items in checklist:
    print(f'[{category}]')
    for item in items:
        print(f'  [ ] {item}')
    print()

print('Ready to compete!')

---
# BONUS SECTION: Advanced Competition Topics

## Evaluation Metrics Deep Dive

Understanding the evaluation metric is non-negotiable. Different metrics reward different model behaviors.

In [ ]:
# EVALUATION METRICS: The Metric Shapes Your Model

print('=== COMMON KAGGLE EVALUATION METRICS ===')
print()
print('CLASSIFICATION:')
print('  AUC-ROC  : Ranking quality. Insensitive to threshold. Use when classes imbalanced.')
print('  Log Loss : Rewards calibrated probabilities. Punishes overconfident wrong predictions.')
print('  F1 Score : Harmonic mean of precision/recall. Use when FP and FN have equal cost.')
print('  MCC      : Matthews Correlation Coefficient. Best for highly imbalanced data.')
print()
print('REGRESSION:')
print('  RMSE     : Heavily penalizes large errors. Sensitive to outliers.')
print('  MAE      : More robust to outliers. Equal weight to all errors.')
print('  RMSLE    : Log-scale RMSE. Use when target spans orders of magnitude.')
print('  R2       : Fraction of variance explained. Interpretable but can mislead.')
print()
print('HOW THE METRIC CHANGES YOUR MODELING STRATEGY:')
print()
print('  RMSE -> Focus on outlier detection and removal')
print('         -> Use robust loss functions (Huber)')
print('         -> Tree models work well (naturally robust to outliers)')
print()
print('  Log Loss -> Calibrate your probabilities (Platt scaling, isotonic regression)')
print('           -> Avoid extreme predictions (0 or 1) - they cause infinite loss')
print('           -> Blend models to get well-calibrated outputs')
print()
print('  AUC-ROC -> Order predictions correctly, not exact values')
print('          -> Rank averaging is particularly effective')
print('          -> Tree-based models excel here')

In [ ]:
# PROBABILITY CALIBRATION - Critical for log loss competitions

from sklearn.calibration import CalibratedClassifierCV, calibration_curve

print('=== PROBABILITY CALIBRATION ===')
print()
print('Tree models (RF, GBM) tend to produce poorly calibrated probabilities.')
print('Logistic regression is usually well-calibrated.')
print()
print('Calibration methods:')
print('  1. Platt Scaling: fit sigmoid on out-of-fold predictions')
print('  2. Isotonic Regression: non-parametric, more flexible')
print()

# Compare calibration
X_cal_data, X_cal_hold, y_cal_data, y_cal_hold = train_test_split(
    X_scaled, y_titanic, test_size=0.3, random_state=99, stratify=y_titanic
)

rf_uncal = RandomForestClassifier(n_estimators=100, random_state=42)
rf_uncal.fit(X_cal_data, y_cal_data)

rf_platt = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    cv=3, method='sigmoid'
)
rf_platt.fit(X_cal_data, y_cal_data)

rf_iso = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    cv=3, method='isotonic'
)
rf_iso.fit(X_cal_data, y_cal_data)

pred_uncal = rf_uncal.predict_proba(X_cal_hold)[:, 1]
pred_platt = rf_platt.predict_proba(X_cal_hold)[:, 1]
pred_iso = rf_iso.predict_proba(X_cal_hold)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for name, preds, color in [
    ('Uncalibrated RF', pred_uncal, 'coral'),
    ('Platt Scaling', pred_platt, 'steelblue'),
    ('Isotonic Reg', pred_iso, 'green')
]:
    fraction_pos, mean_pred = calibration_curve(y_cal_hold, preds, n_bins=10)
    axes[0].plot(mean_pred, fraction_pos, 'o-', label=name, color=color)
    ll = log_loss(y_cal_hold, preds)
    print(f'{name:20s}: LogLoss={ll:.4f}')

axes[0].plot([0,1],[0,1],'k--', linewidth=1, label='Perfect calibration')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of positives')
axes[0].set_title('Calibration Curves')
axes[0].legend(fontsize=9)

# Histogram of predicted probabilities
axes[1].hist(pred_uncal, bins=30, alpha=0.5, label='Uncalibrated', color='coral')
axes[1].hist(pred_platt, bins=30, alpha=0.5, label='Platt', color='steelblue')
axes[1].set_xlabel('Predicted probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Predicted Probability Distributions')
axes[1].legend()

plt.tight_layout()
plt.show()

## Advanced Validation: Stratification and Group Splits

Getting the validation right is the single most important technical decision in a Kaggle competition. Here we cover edge cases that trip up even experienced competitors.

In [ ]:
# ADVERSARIAL VALIDATION - Detect train/test distribution shift

print('=== ADVERSARIAL VALIDATION ===')
print()
print('Problem: Train and test distributions differ (covariate shift).')
print('Solution: Build a classifier to distinguish train from test.')
print()
print('Algorithm:')
print('  1. Label train samples as 0, test samples as 1')
print('  2. Train a binary classifier on combined data')
print('  3. If AUC >> 0.5, distributions are different -> potential problems')
print('  4. Features with high importance = features that shift most')
print('  5. Drop or re-engineer those features')
print()

# Simulate train/test distribution shift
np.random.seed(42)
X_train_adv = np.random.randn(500, 5)
X_test_adv = np.random.randn(300, 5)
X_test_adv[:, 0] += 1.5  # Introduce shift in feature 0
X_test_adv[:, 2] *= 2.0  # Introduce scale shift in feature 2

# Create combined dataset with labels
X_combined = np.vstack([X_train_adv, X_test_adv])
y_combined = np.array([0]*500 + [1]*300)  # 0=train, 1=test

adv_rf = RandomForestClassifier(n_estimators=100, random_state=42)
skf_adv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_scores = cross_val_score(adv_rf, X_combined, y_combined,
                               cv=skf_adv, scoring='roc_auc')

print(f'Adversarial validation AUC: {adv_scores.mean():.4f} +/- {adv_scores.std():.4f}')
print()
if adv_scores.mean() > 0.7:
    print('WARNING: High AUC indicates significant distribution shift!')
    print('Identify and handle shifted features.')
elif adv_scores.mean() > 0.6:
    print('MODERATE shift detected. Investigate feature importances.')
else:
    print('Train and test look similar. No major distribution shift.')

# Find the most shifted features
adv_rf.fit(X_combined, y_combined)
feat_importance_adv = pd.Series(
    adv_rf.feature_importances_,
    index=[f'feature_{i}' for i in range(5)]
).sort_values(ascending=False)
print(f'\nMost shifted features:')
for feat, imp in feat_importance_adv.items():
    bar = '#' * int(imp * 50)
    print(f'  {feat}: {imp:.4f}  {bar}')

In [ ]:
# CROSS-VALIDATION STRATEGIES FOR TIME SERIES

from sklearn.model_selection import TimeSeriesSplit
import pandas as pd

print('=== TIME SERIES CROSS-VALIDATION ===')
print()
print('CRITICAL RULE: Never use random splits on time-series data!')
print('Future data must NEVER appear in training when predicting the past.')
print()

# Create synthetic time-series data
np.random.seed(42)
dates = pd.date_range('2020-01-01', periods=300, freq='D')
ts_data = pd.DataFrame({
    'date': dates,
    'feature1': np.cumsum(np.random.randn(300)) + np.sin(np.arange(300)/30),
    'feature2': np.random.randn(300),
    'target': np.cumsum(np.random.randn(300) * 0.5)
})

X_ts = ts_data[['feature1', 'feature2']].values
y_ts = ts_data['target'].values

print('TimeSeriesSplit configuration:')
tss = TimeSeriesSplit(n_splits=5, gap=7)  # 7-day gap prevents leakage
for fold, (train_idx, val_idx) in enumerate(tss.split(X_ts)):
    print(f'  Fold {fold+1}: train=[{train_idx[0]}, {train_idx[-1]}]  '
          f'val=[{val_idx[0]}, {val_idx[-1]}]  '
          f'(gap={val_idx[0]-train_idx[-1]-1} days)')

print()
print('Time-series feature engineering ideas:')
print('  - Lag features: value at t-1, t-7, t-30')
print('  - Rolling statistics: mean/std over past 7/30/90 days')
print('  - Expanding features: cumulative statistics')
print('  - Date features: day of week, month, quarter, is_holiday')
print('  - Target encoding with time-aware splits')

In [ ]:
# FEATURE IMPORTANCE TYPES - Impurity vs Permutation vs SHAP

print('=== THREE TYPES OF FEATURE IMPORTANCE ===')
print()
print('1. IMPURITY-BASED (default in sklearn tree models):')
print('   Measures how much each feature reduces impurity when split on.')
print('   PROBLEM: Biased toward high-cardinality features.')
print('   PROBLEM: Does not account for feature interactions.')
print()
print('2. PERMUTATION IMPORTANCE:')
print('   Shuffle one feature at a time, measure performance drop.')
print('   More reliable than impurity-based.')
print('   PROBLEM: Slow (one model evaluation per feature per shuffle).')
print('   PROBLEM: Correlated features can have their importance split.')
print()
print('3. SHAP (SHapley Additive exPlanations):')
print('   Game-theoretic approach: fairly allocates importance among features.')
print('   Most reliable and interpretable.')
print('   Shows feature impact per prediction, not just globally.')
print()

# Permutation importance demonstration
from sklearn.inspection import permutation_importance

rf_fi = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fi.fit(X_cal_train_sc, y_cal_train)

# Impurity-based
imp_based = pd.Series(rf_fi.feature_importances_,
                       index=X_titanic.columns)

# Permutation-based
perm_imp = permutation_importance(
    rf_fi, X_cal_val_sc if hasattr(locals().get('X_cal_val_sc', None), '__len__') else X_cal_test_sc,
    y_cal_test, n_repeats=10, random_state=42
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_based.sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Impurity-Based Feature Importance')
axes[0].set_xlabel('Importance')

perm_series = pd.Series(
    perm_imp.importances_mean,
    index=X_titanic.columns
).sort_values()
perm_series.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Permutation Feature Importance (More Reliable)')
axes[1].set_xlabel('Importance Drop When Shuffled')

plt.tight_layout()
plt.show()

print('Note: Permutation importance requires validation set (not training set).')
print('Using validation set prevents inflated importance from memorized patterns.')

In [ ]:
# FIX: Use correct variable name for test set
from sklearn.inspection import permutation_importance

rf_fi2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fi2.fit(X_cal_train, y_cal_train)

imp_based2 = pd.Series(rf_fi2.feature_importances_, index=feature_cols)

perm_imp2 = permutation_importance(
    rf_fi2, X_cal_test, y_cal_test,
    n_repeats=10, random_state=42, n_jobs=-1
)
perm_series2 = pd.Series(perm_imp2.importances_mean, index=feature_cols)

top_n = 10
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_based2.sort_values().tail(top_n).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title(f'Impurity-Based (Top {top_n})')
axes[0].set_xlabel('Importance')

perm_series2.sort_values().tail(top_n).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title(f'Permutation Importance (Top {top_n})')
axes[1].set_xlabel('Mean accuracy drop when shuffled')

plt.tight_layout()
plt.show()

print('Agreement between methods = high confidence in feature importance.')
print('Disagreement = investigate further (possible correlations or scale issues).')

## Handling Imbalanced Data in Competitions

Class imbalance is extremely common in Kaggle competitions (fraud detection, medical diagnosis, etc.). Understanding the right approach is a critical interview topic.

In [ ]:
# IMBALANCED DATA TECHNIQUES

print('=== HANDLING IMBALANCED DATA ===')
print()
print('Common imbalance ratios in competitions:')
print('  Mild imbalance:    85:15 (most competitions)')
print('  Moderate:          95:5  (fraud, medical)')
print('  Severe:            99:1  (rare events)')
print()

# Create imbalanced dataset
from sklearn.datasets import make_classification

X_imb, y_imb = make_classification(
    n_samples=10000, n_features=20, n_informative=10,
    weights=[0.97, 0.03], random_state=42
)

print(f'Imbalanced dataset: {y_imb.sum()} positives out of {len(y_imb)} ({y_imb.mean()*100:.1f}%)')
print()

# Technique 1: Class weight adjustment (simplest)
rf_balanced = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42
)
scores_balanced = cross_val_score(rf_balanced, X_imb, y_imb,
                                    cv=StratifiedKFold(5), scoring='roc_auc')
print(f'RF with class_weight=balanced: AUC={scores_balanced.mean():.4f}')

# Technique 2: No balancing (baseline)
rf_default = RandomForestClassifier(n_estimators=100, random_state=42)
scores_default = cross_val_score(rf_default, X_imb, y_imb,
                                   cv=StratifiedKFold(5), scoring='roc_auc')
print(f'RF with no balancing:          AUC={scores_default.mean():.4f}')

print()
print('Other techniques (not demonstrated to avoid dependencies):')
print('  SMOTE (imbalanced-learn): Synthetic minority oversampling')
print('  ADASYN: Adaptive synthetic sampling')
print('  Undersampling: Remove majority class samples')
print('  Threshold tuning: Adjust decision threshold post-training')
print()
print('Best practices:')
print('  1. Always use StratifiedKFold for imbalanced data')
print('  2. Evaluate with AUC-ROC or PR-AUC, never accuracy')
print('  3. class_weight="balanced" is free and almost always helps')
print('  4. SMOTE should be applied INSIDE the CV loop, not before')

In [ ]:
# HYPERPARAMETER TUNING: PRACTICAL TIPS

print('=== PRACTICAL HYPERPARAMETER TUNING GUIDE ===')
print()
print('GRADIENT BOOSTING parameters (order of importance):')
print()
params = [
    ('n_estimators',     'Medium', 'Start with 300-1000, use early stopping to find optimal'),
    ('learning_rate',    'HIGH',   'Lower = better generalization. Start 0.1, decrease to 0.01-0.05'),
    ('max_depth',        'HIGH',   'Controls tree complexity. 3-6 for GBM, 6-10 for XGBoost'),
    ('subsample',        'Medium', 'Row sampling. 0.7-0.9 usually optimal'),
    ('colsample_bytree', 'Medium', 'Feature sampling. 0.7-0.9 usually optimal'),
    ('min_child_weight', 'Medium', 'Regularization. Increase to reduce overfitting'),
    ('reg_alpha',        'Low',    'L1 regularization. Helps with sparse features'),
    ('reg_lambda',       'Low',    'L2 regularization. Usually keep at 1.0'),
    ('gamma',            'Low',    'Minimum loss reduction to make split. Start at 0'),
]

print(f'{"Parameter":20s} {"Priority":8s} {"Recommendation"}')
print('-' * 80)
for param, priority, rec in params:
    print(f'{param:20s} {priority:8s} {rec}')

print()
print('PRACTICAL TUNING WORKFLOW:')
print('  Step 1: Fix learning_rate=0.1, tune max_depth and min_child_weight')
print('  Step 2: Tune subsample and colsample_bytree')
print('  Step 3: Tune regularization (gamma, alpha, lambda)')
print('  Step 4: Reduce learning_rate, increase n_estimators proportionally')
print('  Step 5: Final fine-tuning with Bayesian optimization')

In [ ]:
# CROSS-VALIDATION SCORE STABILITY ANALYSIS

print('=== CV SCORE STABILITY ANALYSIS ===')
print()
print('A model with high variance in CV scores is unstable.')
print('Do not tune toward unstable models - they will not generalize.')
print()

models_stability = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7),
}

stability_results = {}

print(f'{"Model":25s} {"Mean AUC":10s} {"Std AUC":10s} {"CV of Score":12s}')
print('-' * 60)

skf_stab = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models_stability.items():
    scores = cross_val_score(
        model, X_scaled, y_titanic,
        cv=skf_stab, scoring='roc_auc'
    )
    cv_coef = scores.std() / scores.mean()
    stability_results[name] = scores
    stability = 'STABLE' if scores.std() < 0.02 else 'MODERATE' if scores.std() < 0.04 else 'UNSTABLE'
    print(f'{name:25s} {scores.mean():10.4f} {scores.std():10.4f} {cv_coef:12.4f}  {stability}')

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    [stability_results[m] for m in models_stability],
    labels=list(models_stability.keys()),
    patch_artist=True,
    medianprops={'color': 'red', 'linewidth': 2}
)
colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral', 'lightsalmon']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_ylabel('AUC Score')
ax.set_title('CV Score Distribution Across 10 Folds (Stability Analysis)')
ax.set_xticklabels(list(models_stability.keys()), rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# EXPERIMENT TRACKING - How professionals track progress

print('=== EXPERIMENT TRACKING ===')
print()
print('Professional Kaggle competitors log every experiment.')
print('Without tracking, you cannot reproduce results or identify what worked.')
print()

# Simple experiment tracker
class ExperimentTracker:
    """
    Lightweight experiment tracker for Kaggle competitions.
    In production, use MLflow or Weights & Biases.
    """
    def __init__(self):
        self.experiments = []
        self.best_score = None
        self.best_exp_id = None

    def log(self, exp_id, description, cv_score, lb_score=None, notes=''):
        exp = {
            'id': exp_id,
            'description': description,
            'cv_score': cv_score,
            'lb_score': lb_score,
            'notes': notes
        }
        self.experiments.append(exp)
        if self.best_score is None or cv_score > self.best_score:
            self.best_score = cv_score
            self.best_exp_id = exp_id
        return exp

    def show(self):
        print(f'{"ID":6s} {"CV Score":10s} {"LB Score":10s} {"Description"}')
        print('-' * 70)
        for exp in sorted(self.experiments, key=lambda x: x['cv_score'], reverse=True):
            lb = f"{exp['lb_score']:.4f}" if exp['lb_score'] else 'N/A'
            marker = ' <-- BEST' if exp['id'] == self.best_exp_id else ''
            print(f"{exp['id']:6s} {exp['cv_score']:10.4f} {lb:10s} {exp['description']}{marker}")

# Simulate a competition run
tracker = ExperimentTracker()
tracker.log('EXP01', 'Baseline: LR with raw features', cv_score=0.832)
tracker.log('EXP02', 'Add family_size feature', cv_score=0.841, lb_score=0.839)
tracker.log('EXP03', 'Add fare_per_person feature', cv_score=0.848, lb_score=0.846)
tracker.log('EXP04', 'Add interaction features', cv_score=0.855, lb_score=0.853)
tracker.log('EXP05', 'Switch to GBM', cv_score=0.869, lb_score=0.867)
tracker.log('EXP06', 'Tune GBM (max_depth=4)', cv_score=0.875, lb_score=0.873)
tracker.log('EXP07', 'Add target encoding for embarked', cv_score=0.878, lb_score=0.876)
tracker.log('EXP08', 'Stacking (LR+RF+GBM+SVM+KNN)', cv_score=0.884, lb_score=0.882)
tracker.log('EXP09', 'Overfit attempt (max_depth=10)', cv_score=0.891, lb_score=0.871,
             notes='LB dropped! CV-LB gap widened. Do not use.')
tracker.log('EXP10', 'Rank averaging of EXP06+EXP07+EXP08', cv_score=0.886, lb_score=0.884)

print('Experiment Log:')
tracker.show()
print()
print('Key observation: EXP09 shows CV-LB divergence - sign of overfitting to public LB.')

## Interview Questions and Model Answers

These are common interview questions about Kaggle-style ML that trip up candidates who have only academic experience.

In [ ]:
# COMMON INTERVIEW QUESTIONS AND ANSWERS

qa_pairs = [
    (
        'Q: You have 100 features. How do you approach feature selection?',
        '''A: I use a multi-stage approach:
   Stage 1 - Remove low-variance features (near-constant = no signal)
   Stage 2 - Remove highly correlated features (keep the more interpretable one)
   Stage 3 - Train a quick RF, drop features with near-zero importance
   Stage 4 - Use permutation importance on a validation set (more reliable)
   Stage 5 - RFE with cross-validation to find the optimal subset
   Stage 6 - For regularized linear models, Lasso automatically selects features
   
   Important: Never do feature selection on the full training set.
   Always use cross-validation to avoid selection bias.
   The feature selection step is PART of the CV loop.'''
    ),
    (
        'Q: How do you handle a dataset where train and test have different distributions?',
        '''A: This is covariate shift. My approach:
   1. Run adversarial validation to confirm and quantify the shift
   2. Identify which features shift most (high importance in adversarial model)
   3. Options:
      a. Drop the shifted features (if not predictive on test)
      b. Re-engineer features to be distribution-agnostic
      c. Use importance weighting to re-weight training samples
      d. Domain adaptation techniques
   4. Use a validation strategy that mimics the test distribution
   5. Monitor CV-LB correlation - if poor, the shift is causing problems'''
    ),
    (
        'Q: When would you NOT use stacking?',
        '''A: Several situations where stacking is counterproductive:
   1. Very small datasets (< 1000 samples) - OOF predictions are too noisy
   2. Time constraints - stacking requires K*N model fits
   3. When base models are highly correlated - diversity is key to stacking benefit
   4. Production requirements for low latency - stacking = multiple inference calls
   5. When a single well-tuned model already reaches near-optimal performance
   
   In these cases, simple model averaging or a single well-tuned model is better.'''
    ),
    (
        'Q: Explain the difference between bagging and boosting.',
        '''A: Both are ensemble methods but with different strategies:
   
   BAGGING (e.g., Random Forest):
   - Train models in PARALLEL on bootstrap samples
   - Each model is independent and sees slightly different data
   - Average predictions (reduces variance)
   - Works well when base model has high variance (deep trees)
   - Less prone to overfitting
   
   BOOSTING (e.g., GBM, XGBoost):
   - Train models SEQUENTIALLY, each correcting previous errors
   - Each model focuses on hard examples (high residuals)
   - Weighted sum of predictions (reduces bias)
   - More powerful but more prone to overfitting
   - Requires careful regularization (learning rate, tree depth)'''
    ),
]

for q, a in qa_pairs:
    print(q)
    print(a)
    print()

In [ ]:
# FINAL: Complete performance dashboard

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Kaggle Competition Strategies: Complete Results Dashboard', fontsize=14)

# Plot 1: Titanic classification - stacking improvement
all_model_names = list(model_scores.keys()) + ['STACKING']
all_aucs = [model_scores[n]['AUC'] for n in model_scores] + [stacking_auc]
colors_bar = ['#4C9BE8'] * len(model_scores) + ['#F5A623']

bars = axes[0,0].bar(range(len(all_model_names)), all_aucs, color=colors_bar,
                      edgecolor='black', linewidth=0.5)
axes[0,0].set_xticks(range(len(all_model_names)))
axes[0,0].set_xticklabels(all_model_names, rotation=30, ha='right', fontsize=8)
axes[0,0].set_title('Titanic: Stacking vs Individual Models (AUC)')
axes[0,0].set_ylabel('AUC (higher is better)')
for bar, val in zip(bars, all_aucs):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

# Plot 2: California Housing regression RMSE comparison
reg_names = list(results_reg.keys()) + ['STACKING', 'BASELINE']
reg_rmses = [results_reg[n]['test_rmse'] for n in results_reg] + [meta_rmse, baseline_rmse]
reg_colors = ['#4C9BE8'] * len(results_reg) + ['#F5A623', '#E84C4C']

bars2 = axes[0,1].bar(range(len(reg_names)), reg_rmses, color=reg_colors,
                       edgecolor='black', linewidth=0.5)
axes[0,1].set_xticks(range(len(reg_names)))
axes[0,1].set_xticklabels(reg_names, rotation=30, ha='right', fontsize=8)
axes[0,1].set_title('California Housing: RMSE Comparison')
axes[0,1].set_ylabel('RMSE (lower is better)')

# Plot 3: CV score stability
stab_means = [stability_results[m].mean() for m in models_stability]
stab_stds = [stability_results[m].std() for m in models_stability]
axes[1,0].barh(range(len(models_stability)), stab_means,
                xerr=stab_stds, color='#4C9BE8', ecolor='black',
                capsize=4, edgecolor='black', linewidth=0.5)
axes[1,0].set_yticks(range(len(models_stability)))
axes[1,0].set_yticklabels(list(models_stability.keys()), fontsize=9)
axes[1,0].set_title('Model Stability: Mean CV AUC +/- Std (10 folds)')
axes[1,0].set_xlabel('AUC Score')

# Plot 4: Experiment tracker progress
exp_ids = [e['id'] for e in tracker.experiments]
exp_cv = [e['cv_score'] for e in tracker.experiments]
exp_lb = [e['lb_score'] if e['lb_score'] else None for e in tracker.experiments]

axes[1,1].plot(range(len(exp_cv)), exp_cv, 'o-', color='steelblue',
                label='CV Score', linewidth=2, markersize=6)
lb_vals = [(i, v) for i, v in enumerate(exp_lb) if v is not None]
if lb_vals:
    lb_x, lb_y = zip(*lb_vals)
    axes[1,1].plot(lb_x, lb_y, 's--', color='coral', label='LB Score',
                    linewidth=1.5, markersize=5)
axes[1,1].set_xticks(range(len(exp_ids)))
axes[1,1].set_xticklabels(exp_ids, rotation=45, ha='right', fontsize=7)
axes[1,1].set_title('Experiment Tracker: CV vs LB Score Progression')
axes[1,1].set_ylabel('AUC Score')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Dashboard complete. Ready for competition!')